# Calculate the response

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
from jupyterthemes import jtplot
import matplotlib
import matplotlib.pyplot as plt
import plotly.graph_objects as go

In [ ]:
## Choose the general theme for plots.
plot_theme = "dark"
# plot_theme = 'light'

if plot_theme == "dark":
    plotly_theme = "plotly_dark"
    jtplot_theme = "onedork"
if plot_theme == "light":
    plotly_theme = "plotly"
    jtplot_theme = "grade3"

In [ ]:
two_col_width = 7.056870
one_col_width = 3.4039

two_col_size = (two_col_width, 5.645496)
one_col_size = (one_col_width, 2.7)

# plt.rcParams["font.family"] = "sans-serif"
# plt.rcParams["font.family"] = "Computer Modern Roman"
plt.rcParams["mathtext.fontset"] = "dejavuserif"

font = {"family": "sans-serif", "weight": "normal", "size": 8}

# font = {'family' : 'Charter',
#         'weight' : 'normal',
#         'size'   : 10}

matplotlib.rc("font", **font)
# matplotlib.rc('text', usetex=True)
matplotlib.rc("text.latex", preamble=r"\usepackage{amsmath}")

jtplot.style(
    theme=jtplot_theme,
    context="notebook",
    fscale=1,
    spines=True,
    gridlines=":",
    ticks=True,
    grid=False,
    figsize=(two_col_width, two_col_width / 1.618),
)

matplotlib.rcParams["axes.labelsize"] = 7
matplotlib.rcParams["legend.fontsize"] = 7
matplotlib.rcParams["mathtext.fontset"] = "dejavuserif"

#### Load a network from a file

In [ ]:
from pathlib import Path
from elastory.network.network import Network

network_path = Path("./networks/")
file_path = network_path / "19aef78e1eae444285c22a71a045852d.yaml"

net = Network.load_from_file(file_path)

#### The network data used here is made publicly available in the following [paper](https://www.cell.com/biophysj/biophysj/supplemental/S0006-3495(17)30693-8):
```bibtex
@article{FLECHSIG2017558,
title = "Design of Elastic Networks with Evolutionary Optimized Long-Range Communication as Mechanical Models of Allosteric Proteins",
journal = "Biophysical Journal",
volume = "113",
number = "3",
pages = "558 - 571",
year = "2017",
issn = "0006-3495",
doi = "https://doi.org/10.1016/j.bpj.2017.06.043",
url = "http://www.sciencedirect.com/science/article/pii/S0006349517306938",
author = "Holger Flechsig"
}
```

#### download the data from the supporting information

```bash
curl -O https://www.cell.com/cms/10.1016/j.bpj.2017.06.043/attachment/c2f95c54-548d-42bc-8fe7-9baf626faaf2/mmc11.zip -o ./networks/flechsig/mmc11.zip
curl -O https://www.cell.com/cms/10.1016/j.bpj.2017.06.043/attachment/0fb985e1-0b14-4547-8bc1-32651af77124/mmc12.zip -o ./networks/flechsig/mmc12.zip
curl -O https://www.cell.com/cms/10.1016/j.bpj.2017.06.043/attachment/946ed3a4-e8bf-4aa5-aab1-238f07570659/mmc13.zip -o ./networks/flechsig/mmc13.zip
```
#### Unzip the files
```bash
unzip ./networks/flechsig/mmc11.zip -d ./networks/flechsig/
unzip ./networks/flechsig/mmc12.zip -d ./networks/flechsig/
unzip ./networks/flechsig/mmc13.zip -d ./networks/flechsig/
```

In [ ]:
from rmsd import centroid

from elastory.utils.network_tools import estimate_pocket

path = Path("./networks/flechsig/")

networks = {
    "random": path / "bpj8272mmc11.dat",
    "symmetric": path / "bpj8272mmc12.dat",
    "antisymmetric": path / "bpj8272mmc13.dat",
}

# cutoff in [nm]
cutoff_length = 9

# allosteric site
source = [38, 81]

# active site
target = [149, 189]

nets = {}
for identifier, file_path in networks.items():
    pos = np.loadtxt(file_path)
    pos = pos[:, 1:]
    pos -= centroid(pos)
    net = Network(
        identifier=identifier,
        bead_positions=pos,
        cutoff_length=cutoff_length,
    )

    source_neighbors = []
    for s in source:
        source_neighbors += net.graph.graph.neighbors(s)

    # the response calculation is faster for more than two source beads
    additional_source, source_pairs = estimate_pocket(source_neighbors, pos, 1)
    net.source_pairs = source_pairs + [source]
    net.source = source + additional_source
    net.target = target

    nets[identifier] = net

In [ ]:
from elastory.plot.plotly.utils import plot_network, update_layout

for net in nets.values():
    fig = go.Figure()
    fig = plot_network(fig, net, pulling_connections=False, connections=True)
    fig = update_layout(
        fig,
        hide_axes=True,
        theme=plotly_theme,
        show_legend=False,
    )
    fig.show()

In [ ]:
net = nets["antisymmetric"]

### Calculate the response to pulling source beads towards their shared center of mass

In [ ]:
# choose how far the beads will be pulled, between 0 and 1
# 0 corresponds to all source beads positions coinciding with the center of mass
net.max_pull = 0.7

trajectory = net.calculate_response_concentric(
    preload_steps=250, noise_strength=0, progress_bar=True
)

In [ ]:
from elastory.plot.plotly.utils import response_video, update_layout

fig = go.Figure()
fig = response_video(fig, net, trajectory, speed=10)
fig = update_layout(fig, hide_axes=True, show_legend=False, theme=plotly_theme)
fig.show()

In [ ]:
from elastory.plot.plotly.utils import plot_network_motion, plot_network

fig = go.Figure()
fig = plot_network(fig, net, pulling_connections=False, connections=True)
fig = plot_network_motion(fig=fig, net=net, trajectory=trajectory)
fig = update_layout(
    fig,
    hide_axes=True,
    show_legend=False,
    theme=plotly_theme,
)
fig.show()

### Calculate multiple responses and display avg + std dev in plot

In [ ]:
from tqdm.autonotebook import trange

trajectories = []
for i in trange(25):
    trajectory = net.calculate_response_concentric(
        preload_steps=50, noise_strength=0.03, progress_bar=False
    )
    trajectories.append(trajectory)
trajectories = np.array(trajectories)
trajectories.shape

In [ ]:
fig = go.Figure()
fig = plot_network(fig, net, pulling_connections=False, connections=True)

for trajectory in trajectories:
    fig = plot_network_motion(fig, net, trajectory)

fig = update_layout(
    fig,
    hide_axes=True,
    theme=plotly_theme,
    show_legend=False,
)
fig.show()

#### Proxy for Closing of Pockets: Radius of Gyration

In [ ]:
from elastory.response.observables import rgyr

In [ ]:
# calculate rgyr for source and target beads (mean and std dev)
rgyrs_source = [rgyr(traj, np.array(net.source)) for traj in trajectories]
rgyrs_target = [rgyr(traj, np.array(net.target)) for traj in trajectories]

rgyr_source_mean = np.mean(rgyrs_source, axis=0)
rgyr_target_mean = np.mean(rgyrs_target, axis=0)
rgyr_source_std = np.std(rgyrs_source, axis=0)
rgyr_target_std = np.std(rgyrs_target, axis=0)

upper_bound_x = rgyr_source_mean + rgyr_source_std
lower_bound_x = rgyr_source_mean - rgyr_source_std

upper_bound_y = rgyr_target_mean + rgyr_target_std
lower_bound_y = rgyr_target_mean - rgyr_target_std

In [ ]:
from plotly.io import to_image
from io import BytesIO
from PIL import Image

In [ ]:
# 3D plotly (network)
plotly_fig = go.Figure()
plotly_fig = plot_network(plotly_fig, net, pulling_connections=False, connections=True)
for trajectory in trajectories:
    plotly_fig = plot_network_motion(fig=plotly_fig, net=net, trajectory=trajectory)
plotly_fig = update_layout(
    plotly_fig,
    hide_axes=True,
    show_legend=False,
    theme=plotly_theme,
)
plotly_fig.update_layout(
    paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)", margin=dict(l=0, r=0, t=0, b=0)
)

# Convert Plotly figure to image
img_bytes = to_image(plotly_fig, format="png", scale=2)

# Convert bytes to numpy array with alpha channel
img = Image.open(BytesIO(img_bytes)).convert("RGBA")
img_array = np.array(img)

In [ ]:
from elastory.plot.utils import axes_formatter

In [ ]:
# 2D matplotlib (rgyr)
fig, ax = plt.subplots(
    1,
    1,
    figsize=(two_col_width, two_col_width / 1.618),
    dpi=300,
    constrained_layout=True,
    gridspec_kw={"width_ratios": [1]},
    sharey=False,
)
axes_formatter([ax])


ax.plot(
    rgyr_source_mean,
    rgyr_target_mean,
    ls="-",
    marker="o",
    ms=3,
    alpha=0.95,
    color="darkviolet",
    label="full response",
)
ax.errorbar(
    rgyr_source_mean,
    rgyr_target_mean,
    xerr=rgyr_source_std,
    yerr=rgyr_target_std,
    ls=None,
    marker=None,
    ms=0,
    fmt="o",
    ecolor="gray",
    alpha=0.6,
    label="Error bars",
)

ax.fill_betweenx(
    rgyr_target_mean,
    lower_bound_x,
    upper_bound_x,
    alpha=0.3,
    label="std dev (source)",
    color="darkblue",
)
ax.fill_between(
    rgyr_source_mean,
    lower_bound_y,
    upper_bound_y,
    alpha=0.1,
    label="std dev (target)",
    color="darkred",
)

ax.set_ylabel("$r_{\\rm gyr}(T)$", fontsize=15)
ax.set_xlabel("$r_{\\rm gyr}(S)$", fontsize=15)
ax.legend(fontsize=12)

# insert the plotly image in the matplotlib plot
# Calculate the size and position for the subplot
subplot_width = 0.5
subplot_height = 0.5
subplot_x = 0.03
# subplot_y = 1 - subplot_height - 0.03
subplot_y = 0.1

# Create a new axes for the Plotly subplot
subplot_ax = fig.add_axes(rect=(subplot_x, subplot_y, subplot_width, subplot_height))

# Embed Plotly image in the subplot
subplot_ax.imshow(img_array)
subplot_ax.axis("off")

plt.show()

In [ ]:
from elastory.utils.hessian import calc_hessian, calc_hessian_with_nzeqd
from elastory.utils.misc import degrees_of_freedom

dof = degrees_of_freedom(net.D)

In [ ]:
from scipy.linalg import eigh

pos = net.geometry.bead_positions
pos = trajectory[15]

# hessian with zero equilibrium distances?
hessian_zeq, *_ = calc_hessian(
    pos=pos,
    laplacian=net.laplacian,
)

# hessian with non-zero equilibrium distances
hessian_nzeq = calc_hessian_with_nzeqd(
    eq_dist=net.geometry.eq_dist,
    pos=pos,
    connected=net.connected,
)

eigvals = eigh(hessian_nzeq, eigvals_only=True)  # , subset_by_index=([dof, n - 1]))
plt.plot(eigvals)
eigvals = eigh(hessian_zeq, eigvals_only=True)  # , subset_by_index=([dof, n - 1]))
plt.plot(eigvals)

In [ ]:
from elastory.response.observables import eigvals_and_eigvecs

eigvals_zeq = []
eigvals_nzeq = []
eigvecs_zeq = {i: [] for i in range(dof + 1)}
eigvecs_nzeq = {i: [] for i in range(dof + 1)}

pos = net.geometry.bead_positions

# hessian with zero equilibrium distances?
hessian_zeq, *_ = calc_hessian(
    pos=pos,
    laplacian=net.laplacian,
)

# hessian with non-zero equilibrium distances
hessian_nzeq = calc_hessian_with_nzeqd(
    eq_dist=net.geometry.eq_dist,
    pos=pos,
    connected=net.connected,
)

w0_zeq, v0_zeq = eigvals_and_eigvecs(net=net, hessian=hessian_zeq)
w0_nzeq, v0_nzeq = eigvals_and_eigvecs(net=net, hessian=hessian_nzeq)

num_eigvecs = 5
for i in trange(1, len(trajectory)):
    pos = trajectory[i]

    # hessian with zero equilibrium distances?
    hessian_zeq, *_ = calc_hessian(
        pos=pos,
        laplacian=net.laplacian,
    )

    # hessian with non-zero equilibrium distances
    hessian_nzeq = calc_hessian_with_nzeqd(
        eq_dist=net.geometry.eq_dist,
        pos=pos,
        connected=net.connected,
    )

    wi_zeq, vi_zeq = eigvals_and_eigvecs(net=net, hessian=hessian_zeq)
    wi_nzeq, vi_nzeq = eigvals_and_eigvecs(net=net, hessian=hessian_nzeq)
    eigvals_zeq.append(wi_zeq)
    eigvals_nzeq.append(wi_nzeq)

    for n in range(dof + 1):
        eigvecs_zeq[n].append(np.abs(vi_zeq[:, n] @ v0_zeq[:, n]))
        eigvecs_nzeq[n].append(np.abs(vi_nzeq[:, n] @ v0_nzeq[:, n]))

eigvals_zeq = np.array(eigvals_zeq).T
eigvals_nzeq = np.array(eigvals_nzeq).T

In [ ]:
fig, (ax0, ax1) = plt.subplots(
    1,
    2,
    figsize=(two_col_width, one_col_width / 1.618),
    dpi=300,
    gridspec_kw={"width_ratios": [5, 4]},
    constrained_layout=True,
)
axes_formatter([ax0, ax1])

for i in range(num_eigvecs + 1):
    ax0.plot(eigvecs_zeq[i], ".-", ms=4, lw=0.75, label="$k=$ " + str(i))
    ax1.plot(eigvals_zeq[i], ".-", ms=3.5, lw=0.75, label="$k=$ " + str(i))

ax0.legend(ncol=2)
# ax1.legend()2

ax0.set_xlabel("steps ~ i=1\\dots $\\Omega$, ~ $\\Omega=100 $")
ax1.set_xlabel("steps ~ i=1\\dots $\\Omega$, ~ $\\Omega=100 $")

ax0.set_ylabel(
    "$\\alpha_k^{(i)} \
                = \\langle \\boldsymbol{v}_k^{(0)} | \\boldsymbol{v}_k^{(i)} \\rangle$",
    fontsize=10,
)

ax1.set_ylabel("$ \\lambda_k^{(i)}$", fontsize=10)
# plt.savefig('/home/mav/projects/diss-latex/content/figures/evec/proj.png',
#             dpi=300)

plt.show()

In [ ]:
fig, (ax0, ax1) = plt.subplots(
    1,
    2,
    figsize=(two_col_width, one_col_width / 1.618),
    dpi=300,
    gridspec_kw={"width_ratios": [5, 4]},
    constrained_layout=True,
)
axes_formatter([ax0, ax1])

for i in range(num_eigvecs + 1):
    ax0.plot(eigvecs_nzeq[i], ".-", ms=4, lw=0.75, label="$k=$ " + str(i))
    ax1.plot(eigvals_nzeq[i], ".-", ms=3.5, lw=0.75, label="$k=$ " + str(i))

ax0.legend(ncol=2)
# ax1.legend()2

ax0.set_xlabel("steps ~ i=1\\dots $\\Omega$, ~ $\\Omega=100 $")
ax1.set_xlabel("steps ~ i=1\\dots $\\Omega$, ~ $\\Omega=100 $")

ax0.set_ylabel(
    "$\\alpha_k^{(i)} \
                = \\langle \\boldsymbol{v}_k^{(0)} | \\boldsymbol{v}_k^{(i)} \\rangle$",
    fontsize=10,
)

ax1.set_ylabel("$ \\lambda_k^{(i)}$", fontsize=10)
# plt.savefig('/home/mav/projects/diss-latex/content/figures/evec/proj.png',
#             dpi=300)

plt.show()

# WIP

#### Eigenvalues during the response

#### Eigenvectors during the response

#### Potential Energy

#### Linear Response

#### Reciprocal Response

#### Repulsive Beads
##### Repulsive potential parameters for response
below this distance a repulsive potential can be applied

In [ ]:
repulsive_cutoff = 0.1

----

#### More Visualization

In [ ]:
from pathlib import Path

visualization_path = Path("./visualizations/")
visualization_path.mkdir(exist_ok=True)

In [ ]:
fig = go.Figure()
fig = plot_network(fig, net, pulling_connections=False, connections=True)

for trajectory in trajectories:
    fig = plot_network_motion(fig, net, trajectory)

fig = update_layout(
    fig,
    hide_axes=True,
    theme=plotly_theme,
    show_legend=False,
)

In [ ]:
# Export as interactive HTML
fig.write_html(
    visualization_path / "3d_plot_interactive.html", full_html=False, include_plotlyjs="cdn"
)

In [ ]:
# Create (high-resolution) PNG
fig.update_layout(
    scene=dict(aspectmode="cube"),
    width=1200,  # Increased width
    height=1200,  # Increased height
)

fig.write_image(visualization_path / "3d_plot.png", width=1200, height=1200, scale=4)

In [ ]:
# Create rotating GIF
from PIL import Image
import io
import numpy as np
from tqdm.autonotebook import tqdm


def ease_in_out(t):
    return 0.5 * (1 - np.cos(np.pi * t))

In [ ]:
frames = []
num_frames = 120  # Increase number of frames for smoother animation

for i in tqdm(range(num_frames)):
    t = i / (num_frames - 1)

    # Create a full cycle: 0 to 1 to 0
    cycle = 1 - abs(2 * t - 1)

    angle = 30 * (ease_in_out(cycle) - 0.5)  # Range from -15 to +15 degrees

    # fig.update_layout(scene_camera=dict(eye=dict(x=np.cos(np.radians(angle)),
    #                                              y=np.sin(np.radians(angle)),
    #                                              z=0.5)))

    fig.update_layout(
        scene=dict(aspectmode="cube"),
        width=1200,  # Increased width
        height=1200,  # Increased height
    )
    img_bytes = fig.to_image(format="png", width=1200, height=1200, scale=4)
    frames.append(Image.open(io.BytesIO(img_bytes)))

# Save the GIF
frames[0].save(
    visualization_path / "rotating_3d_plot.gif",
    save_all=True,
    append_images=frames[1:],
    duration=50,  # Duration for each frame in milliseconds
    loop=0,  # 0 means loop indefinitely
)

In [ ]:
from scipy.ndimage import gaussian_filter1d


def create_std_dev_surface(
    trajectories, net, bead_index, num_time_points=20, num_circle_points=16, smoothing_sigma=2
):
    mean_trajectory = np.mean(trajectories, axis=0).astype(np.float64)
    std_trajectory = np.std(trajectories, axis=0).astype(np.float64)

    # Apply Gaussian smoothing
    mean_trajectory_smooth = gaussian_filter1d(
        mean_trajectory[:, bead_index], sigma=smoothing_sigma, axis=0
    )
    std_trajectory_smooth = gaussian_filter1d(
        std_trajectory[:, bead_index], sigma=smoothing_sigma, axis=0
    )

    # Downsample the smoothed trajectory
    time_indices = np.linspace(0, mean_trajectory_smooth.shape[0] - 1, num_time_points, dtype=int)
    mean_traj = mean_trajectory_smooth[time_indices]
    std_traj = std_trajectory_smooth[time_indices]

    theta = np.linspace(0, 2 * np.pi, num_circle_points)

    # Calculate trajectory direction
    dx = np.gradient(mean_traj[:, 0])
    dy = np.gradient(mean_traj[:, 1])
    dz = np.gradient(mean_traj[:, 2])

    # Normalize direction vectors
    norm = np.sqrt(dx**2 + dy**2 + dz**2)
    dx, dy, dz = dx / norm, dy / norm, dz / norm

    x_surface = np.zeros((num_time_points, num_circle_points))
    y_surface = np.zeros((num_time_points, num_circle_points))
    z_surface = np.zeros((num_time_points, num_circle_points))

    for i in range(num_time_points):
        perpx = np.array([1.0, 0.0, 0.0])
        perpx -= np.dot(perpx, [dx[i], dy[i], dz[i]]) * np.array([dx[i], dy[i], dz[i]])
        perpx /= np.linalg.norm(perpx)
        perpy = np.cross([dx[i], dy[i], dz[i]], perpx)

        circle_points = (
            std_traj[i, 0] * np.cos(theta)[:, np.newaxis] * perpx
            + std_traj[i, 1] * np.sin(theta)[:, np.newaxis] * perpy
        )

        x_surface[i] = mean_traj[i, 0] + circle_points[:, 0]
        y_surface[i] = mean_traj[i, 1] + circle_points[:, 1]
        z_surface[i] = mean_traj[i, 2] + circle_points[:, 2]

    return x_surface, y_surface, z_surface, mean_traj

In [ ]:
fig = go.Figure()
fig = plot_network(fig, net, pulling_connections=False, connections=True)
# fig = plot_network_with_motion(fig, net, mean_trajectory)

assert net.source
assert net.target

for bead in range(net.N):
    if bead in net.source:
        colorscale = [[0, "rgb(200,230,255)"], [1, "rgb(100,130,255)"]]  # blue
    elif bead in net.target:
        colorscale = [[0, "rgb(255,200,200)"], [1, "rgb(255,100,100)"]]  # red
    else:
        colorscale = [[0, "rgb(0,100,0)"], [1, "rgb(0,200,0)"]]  # green

    x_surface, y_surface, z_surface, mean_traj = create_std_dev_surface(
        trajectories, net, bead, num_time_points=10, num_circle_points=10, smoothing_sigma=2
    )

    # Create the surface
    fig.add_trace(
        go.Surface(
            x=x_surface,
            y=y_surface,
            z=z_surface,
            colorscale=colorscale,
            opacity=0.7,
            showscale=False,
            name=f"Bead {bead} Std Dev",
        )
    )


fig = update_layout(fig, hide_axes=True, show_legend=True, theme=plotly_theme)
fig.show()

fig.write_html(
    visualization_path / "3d_std_dev_surf_interactive.html",
    full_html=False,
    include_plotlyjs="cdn",
)